## Sandbox  - Get Data and Adminstrative Mapping

#### Content  
1. Get Data From DB
    - StockPoints and Coordinations
    - StockPoints Mapping (State, LGA, LCDA)
    - Customer DIM (AddressBook)
    - Active Stock Point Customers (>= 2025-01-01)

2. Load Data from Local Repo
2. Get Adminstrative 2, 3 Mapping
    - LGA and LCDA (Wards)

In [ ]:
import pandas as pd
import numpy as np
import h3  
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip 
from folium.plugins import FastMarkerCluster, HeatMap 
from shapely.geometry import Polygon, Point  
# pd.set_option('display.max.rows', None)
pd.set_option('display.max.columns', None)

#### 1. Fetch Data from DB

In [ ]:
import pandas as pd
import pyodbc  # or sqlalchemy, depending on your setup
from codebase.database.get_connection import get_connection_string, get_connection
from codebase.utils.utils import setup_logging 
import warnings
# from jinja2 import Template

In [ ]:
logger = setup_logging(log_dir='log-sandbox-sp-clustering-and-routing', projname='log-sandbox-sp-clustering-and-routing')

In [ ]:
# Get Connection
repl_con_string = get_connection_string(logger = logger, database='VconnectMasterDWR', server_type='replica') 

In [ ]:
# conn.close()

In [ ]:
## Stock Point Dim: Stock_Point_ID	Stock_point_Name	Lattitude	Longitude
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_dim.sql", 'r') as file:
            sql_query = file.read()
        df_sp_dim = pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_dim.empty:
        df_sp_dim.to_feather('./input/df_sp_dim.feather')
except Exception as e:
    logger.error(f'Error fetch stock_point_dim:\n{e}')
    

In [ ]:
## Location and Stockpoint Mapping
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_location_map.sql", 'r') as file:
            sql_query = file.read()
        df_sp_location_mapping= pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_location_mapping.empty:
        df_sp_location_mapping.to_feather('./input/df_sp_location_mapping.feather')
except Exception as e:
    logger.error(f'Error fetch stock point location mapping:\n{e}')
    

In [ ]:
## Customer Dim
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/get_customer_dim.sql", 'r') as file:
            sql_query = file.read()
        df_customer_dim= pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_location_mapping.empty:
        df_customer_dim.to_feather('./input/df_customer_dim.feather')
except Exception as e:
    logger.error(f'Error fetching customer dim from db:\n{e}')
    

In [ ]:
## Active Stock Point Customer '2025-01-01' till date
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_active_customers.sql", 'r') as file:
            sql_query = file.read()
        df_sp_active_customers= pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_location_mapping.empty:
        df_sp_active_customers.to_feather('./input/df_sp_active_customers.feather')
except Exception as e:
    logger.error(f'Error fetching sp active customers dim from db:\n{e}')

In [ ]:
# df_sp_active_customers.groupby('Stock_Point_ID')['CustomerID'].count()

#### 2. LOAD LOCAL DATA

In [ ]:
df_sp_dim = pd.read_feather('./input/df_sp_dim.feather')
df_sp_active_customers = pd.read_feather('./input/df_sp_active_customers.feather')
df_customer_dim = pd.read_feather('./input/df_customer_dim.feather')
df_sp_location_mapping = pd.read_feather('./input/df_sp_location_mapping.feather')

In [ ]:
df_customer_dim = (
    df_customer_dim.copy()
    .assign(Address=lambda df: df['FullAddress'].fillna(df['Location']))
    .assign(IsLocationCaptured=lambda df: df['IsLocationCaptured'].fillna(0))
    .assign(IsLocationVerified=lambda df: df['IsLocationVerified'].fillna(0))
    .assign(Address=lambda df: df.apply(
        lambda row: row['Location'] if row['Address'] == '' and row['Location'] != '' else row['Address'],
        axis=1
    ))
    .assign(IsLocationCaptured=lambda df: df.apply(
        lambda row: 0 if row['IsLocationCaptured'] == 1 and pd.isna(row['LocationSubmittedDate']) else row['IsLocationCaptured'],
        axis=1
    ))
    .assign(IsLocationVerified=lambda df: df.apply(
        lambda row: 0 if row['IsLocationVerified'] == 1 and pd.isna(row['LocationVerifiedDate']) else row['IsLocationVerified'],
        axis=1
    ))
    .drop(columns = ['ContentID',  'BusinessID',  'IsLocationSubmitted', 'LocationSubmittedDate',  
                     'LocationVerifiedDate', 'Location', 'FullAddress','rn_active' ])
)
 

#### 3. GeoJSON files of Nigeria Administrative Levels (LGA, Wards)  
- https://gadm.org/download_country.html
- https://data.grid3.org/datasets/GRID3::grid3-nga-operational-lga-boundaries/about
- https://data.grid3.org/datasets/GRID3::grid3-nga-operational-wards-v1-0/about
- https://data.grid3.org/datasets/GRID3::grid3-nga-markets/


In [ ]:
## GeoJson to PD
import geopandas as gpd  

ng_admin_lga_gdf = (gpd.read_file('../input/geojson/GRID3_NGA_-_Operational_LGA_Boundaries.geojson')
                     .drop(columns=['FID', 'globalid', 'uniq_id', 'timestamp', 'editor'])
                    )

ng_admin_ward_gdf = (gpd.read_file('../input/geojson/Nigeria_-_Ward_Boundaries.geojson')
                     .drop(columns=['FID', 'globalid', 'uniq_id', 'timestamp', 'editor'])
                    )

ng_admin_ward_gdf.columns = ng_admin_ward_gdf.columns.str.lower()

cols_ngwards = ['statename', 'lganame', 'lgacode','wardname', 'wardcode' ]

ng_admin_ward_gdf[cols_ngwards] = ng_admin_ward_gdf[cols_ngwards].apply(lambda x: x.str.lower()) 

# Display the DataFrame
print(ng_admin_lga_gdf.shape[0])
print(ng_admin_ward_gdf.shape[0])
# print(ng_admin_gdf.columns)
ng_admin_ward_gdf.sample(2)


In [ ]:
print(ng_admin_lga_gdf.crs)
print(ng_admin_ward_gdf.crs)

#### 4. Filter Data

In [ ]:
spid_causeway = 1647113
dict_causeway = {}

# SP DIM
dict_causeway['sp_dim'] = df_sp_dim.query(f'Stock_Point_ID == {spid_causeway}')

# Active Customer DIm
dict_causeway['sp_active_customers_dim'] = (df_sp_active_customers
                                                .query(f'Stock_Point_ID == {spid_causeway}')
                                                .merge(df_customer_dim, on='CustomerID', how='inner') 
                                                )
  

dict_causeway['sp_mapped_location'] = (df_sp_location_mapping.query('Stock_Point_ID == 1647113')
                       .query('~LCDA_Name.str.contains("self|push")', engine='python')
                       .drop(columns=['Region','State_ID'])
                       .reset_index(drop=True)
                      )

dict_causeway['df_ng_ward_lagos'] = (ng_admin_ward_gdf.query('statename == "lagos"') 
                                        .reset_index(drop=True)
                                        )

In [ ]:
import pickle

# Save the dictionary to a file
with open('./input/dict_causeway.pickle', 'wb') as file:
    pickle.dump(dict_causeway, file)


In [ ]:
# dict_causeway.keys()
# 'sp_dim', 'sp_active_customers_dim', 'sp_mapped_location'

# df_ng_ward_lagos.columns
# ['wardname', 'wardcode', 'lganame', 'lgacode', 'statename', 'statecode',
#  'amapcode', 'status', 'source', 'urban', 'shape__area', 'shape__length', 'geometry']

#### 5. Data Clearning Task: Fill Ward Name for LCDA or SP-Location Mapping

In [ ]:
# # cols_ngwards = ['statename', 'lganame', 'lgacode','wardname', 'wardcode' ]

# # df_ng_ward = ng_admin_ward_gdf[cols_ngwards].apply(lambda x: x.str.lower()) 

# # df_ng_ward.columns = {f'{col+"_ng"}' for col in df_ng_ward.columns}

# # df_ng_ward.columns

# # to lower case
# cols_to_lower = ['State_Name', 'LGA_Name', 'LCDA_Name']
# df_sp_location_mapping[cols_to_lower] = df_sp_location_mapping[cols_to_lower].apply(lambda x: x.str.lower())


# df_causeway_mapping = (df_sp_location_mapping.query('Stock_Point_ID == 1647113')
#                        .query('~LCDA_Name.str.contains("self|push")', engine='python')
#                        .drop(columns=['Region','State_ID'])
#                        .reset_index(drop=True)
#                       )


# df_ng_ward_lagos = (df_ng_ward.query('statename_ng == "lagos"') 
#                        .reset_index(drop=True)
#                       )

# df_causeway_mapping['wardname_ng'] = None
# df_causeway_mapping['wardcode_ng'] = None

# print(df_ng_ward_lagos.head(3))
# df_causeway_mapping.head(3) 

# # Create a Pandas Excel writer using openpyxl as the engine
# with pd.ExcelWriter('./output/CAUSEWAY_MAPPING_WARD.xlsx', engine='openpyxl') as writer:
#     # Write each DataFrame to a different worksheet
#     df_causeway_mapping.to_excel(writer, sheet_name='causeway_location_mapping', index=False)
#     df_ng_ward_lagos.to_excel(writer, sheet_name='ng_wards', index=False)

In [ ]:

# df_ng_ward_lagos.head(10)

## 2. Sandbox - H3 GRID (HEIRARCHICAL HEXAGON SPATIAL INDEXING)

Content  
1. Building a h3process class

#### H3PROCESSOR CLASS

Methods  
1. process_points
2. aggregate_by_h3
3. get_cell_geometry
4. process_large_dataset
5. memory_efficient_aggregation

In [ ]:
# !pip install h3 
# !pip install --upgrade h3

In [ ]:
import h3
print(h3.__version__) 

In [ ]:
import h3

lat, lng = 37.769377, -122.388903
resolution = 9

h3_index = h3.latlng_to_cell(lat=lat, lng=lng, res=resolution)
print(h3_index)

In [ ]:
import h3
import pandas as pd

class H3Processor:
    def __init__(self, resolution=9):
        self.resolution = resolution

    @staticmethod
    def _safe_latlng_to_cell(lat, lng, resolution):
        """Safely convert coordinates to H3 with validation using v4.x function."""
        try:
            # Validate coordinates
            if not (-90 <= lat <= 90 and -180 <= lng <= 180):
                raise ValueError(f"Invalid coordinates: {lat}, {lng}")

            # Validate resolution
            if not (0 <= resolution <= 15):
                raise ValueError(f"Invalid resolution: {resolution}")

            return h3.latlng_to_cell(lat, lng, resolution)

        except Exception as e:
            print(f"Error converting coordinates: {e}")
            return None
   
    def process_points(self, df, lat_col='lat', lng_col='lng'):
        """Convert DataFrame points to H3 indexes using v4.x function."""
        # Using the safe conversion method
        df['h3_index'] = df.apply(
            lambda row: self._safe_latlng_to_cell(lat=row[lat_col], lng=row[lng_col], resolution=self.resolution),
            axis=1
        )
        return df

    def aggregate_by_h3(self, df, value_col, agg_func='sum'):
        """Aggregate values by H3 cell."""
        # Ensure 'h3_index' exists and is appropriate for grouping
        if 'h3_index' not in df.columns:
            print("Warning: 'h3_index' column not found. Run process_points first.")
            return pd.DataFrame() # Return empty DataFrame or raise error

        return df.groupby('h3_index')[value_col].agg(agg_func).reset_index()

    def get_cell_geometry(self, h3_index):
        """Get cell center and boundary using v4.x functions."""
        # Check if h3_index is valid before proceeding
        if not h3.is_valid_cell(h3_index):
            print(f"Invalid H3 index: {h3_index}")
            return None

        return {
            'center': h3.cell_to_latlng(h3_index),
            'boundary': h3.cell_to_boundary(h3_index)
        }
    
    @staticmethod
    def process_large_dataset(df, resolution=9, batch_size=10000):
        """Process large datasets in batches using v4.x function."""
        results = []

        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size].copy()
            batch['h3_index'] = batch.apply(
                lambda row: h3.latlng_to_cell(row['lat'], row['lng'], resolution),
                axis=1
            )
            results.append(batch)

        return pd.concat(results, ignore_index=True)

    @staticmethod
    def memory_efficient_aggregation(df, value_col='value'):
        """Aggregate large datasets efficiently. Requires 'h3_index' and 'value_col' to exist."""
        # Use categorical data type for H3 indexes to save memory
        if 'h3_index' not in df.columns:
            raise ValueError("'h3_index' column not found for aggregation.")
        if value_col not in df.columns:
            raise ValueError(f"Value column '{value_col}' not found for aggregation.")
            
        df['h3_index'] = df['h3_index'].astype('category')

        return df.groupby('h3_index', observed=True).agg({
            value_col: ['sum', 'count', 'mean']
        }).reset_index()

In [ ]:

# Usage example
processor = H3Processor(resolution=9)
data = pd.DataFrame({
    'lat': [37.7749, 37.7849, 37.7649],
    'lng': [-122.4194, -122.4094, -122.4294],
    'sales': [100, 200, 150]
})

# Process data
processed = processor.process_points(data)
aggregated = processor.aggregate_by_h3(processed, 'sales')
print(aggregated)

#### H3ProcessorV4

In [ ]:
class H3ProcessorV4:
    """Modern H3 processor using v4 API"""

    def __init__(self, resolution=9):
        self.resolution = resolution

    def process_points(self, df, lat_col='lat', lng_col='lng'):
        """Convert DataFrame points to H3 indexes (NEW API)"""
        df = df.copy()
        df['h3_index'] = df.apply(
            lambda row: h3.latlng_to_cell(row[lat_col], row[lng_col], self.resolution),
            axis=1
        )
        return df
    
    # @staticmethod
    def process_large_dataset(self, df, lat_col='lat', lng_col='lng', batch_size=10000, resolution=None):
        """Process large datasets in batches using v4.x function."""
        results = []
        resolution_ = resolution if resolution is not None else self.resolution
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size].copy()
            batch['h3_index'] = batch.apply(
                lambda row: h3.latlng_to_cell(row[lat_col], row[lng_col], resolution_),
                axis=1
            )
            results.append(batch)

        return pd.concat(results, ignore_index=True)    
    
    def aggregate_by_h3(self, df, value_col, agg_func='sum'):
        """Aggregate values by H3 cell (NEW API)"""
        result = df.groupby('h3_index')[value_col].agg(agg_func).reset_index()

        # Add geometry information
        result['center_lat'] = result['h3_index'].apply(lambda x: h3.cell_to_latlng(x)[0])
        result['center_lng'] = result['h3_index'].apply(lambda x: h3.cell_to_latlng(x)[1])
        result['resolution'] = result['h3_index'].apply(lambda x: h3.get_resolution(x))

        return result

    def get_cell_info(self, h3_index):
        """Get comprehensive cell information (NEW API)"""
        if not h3.is_valid_cell(h3_index):
            return None

        return {
            'h3_index': h3_index,
            'center': h3.cell_to_latlng(h3_index),
            'boundary': h3.cell_to_boundary(h3_index),
            'resolution': h3.get_resolution(h3_index),
            'is_pentagon': h3.is_pentagon(h3_index),
            'area_km2': h3.cell_area(h3_index, 'km^2'),
            'base_cell': h3.get_base_cell_number(h3_index)
        }
    
    def get_neighbors_info(self, h3_index, k=1):
        """Get neighbors with additional information (NEW API)"""
        neighbors = h3.grid_disk(h3_index, k)
        neighbors_info = []

        center_coord = h3.cell_to_latlng(h3_index)

        for neighbor in neighbors:
            neighbor_coord = h3.cell_to_latlng(neighbor)
            distance = h3.great_circle_distance(center_coord, neighbor_coord, 'km')
            grid_distance = h3.grid_distance(h3_index, neighbor)

            neighbors_info.append({
                'h3_index': neighbor,
                'center': neighbor_coord,
                'distance_km': distance,
                'grid_distance': grid_distance,
                'is_center': neighbor == h3_index
            })

        return neighbors_info

    def hierarchical_operations(self, h3_index):
        """Demonstrate hierarchical operations (NEW API)"""
        current_res = h3.get_resolution(h3_index)

        operations = {
            'current_cell': h3_index,
            'current_resolution': current_res,
            'parent_cells': {},
            'children_cells': {},
            'center_child': None
        }

        # Get parents at different resolutions
        for res in range(current_res):
            try:
                parent = h3.cell_to_parent(h3_index, res)
                operations['parent_cells'][res] = parent
            except:
                break

        # Get children at different resolutions
        for res in range(current_res + 1, min(current_res + 3, 16)):
            try:
                children = h3.cell_to_children(h3_index, res)
                operations['children_cells'][res] = list(children)
                if res == current_res + 1:
                    operations['center_child'] = h3.cell_to_center_child(h3_index, res)
            except:
                break

        return operations

    def recommend_resolution_for_radius(self,min_radius_km, max_radius_km, verbose =False):
        """
        Recommend H3 resolution based on a desired radius range in kilometers.
        The "radius" of a hexagon is approximated by its average edge length.
        """
        best_resolution = None
        best_deviation = float('inf')

        # Iterate over all possible H3 resolutions (0 to 15)
        for res in range(16):
            avg_edge_length_km = h3.average_hexagon_edge_length(res, unit='km')
            radius_km = avg_edge_length_km

            # Calculate deviation from the target range
            if radius_km < min_radius_km:
                deviation = min_radius_km - radius_km
            elif radius_km > max_radius_km:
                deviation = radius_km - max_radius_km
            else:
                deviation = 0

            # Check if this resolution is closer to the desired range
            if deviation < best_deviation:
                best_deviation = deviation
                best_resolution = res

        if verbose:
            if best_resolution is not None:
                print(f"Recommended H3 resolution for radius {min_radius_km}-{max_radius_km} km: {best_resolution}")
            else:
                print(f"No suitable H3 resolution found for the radius range {min_radius_km}-{max_radius_km} km.")
                
        return best_resolution

    def recommend_resolution(self,area_km2, purpose="analysis"):
        """Recommend H3 resolution based on area and purpose."""

        # Define thresholds based on purpose
        purpose_thresholds = {
            "analysis": 100,  # Higher detail for analysis
            "visualization": 50,  # Moderate detail for visualization
            "overview": 10  # Lower detail for overview
        }

        # Get the threshold based on the purpose, default to analysis if purpose not found
        threshold = purpose_thresholds.get(purpose, purpose_thresholds["analysis"])

        # Get resolution cell areas
        resolutions = {}
        for res in range(16):
            try:
                cell_area = h3.average_hexagon_area(res, 'km^2')
                if cell_area is not None:
                    resolutions[res] = cell_area
            except Exception as e:
                print(f"Error calculating area for resolution {res}: {e}")
                continue

        # Find resolution where cell size is appropriate for the area
        for res, cell_area in resolutions.items():
            if area_km2 / cell_area >= threshold:
                return res

        # Return the highest resolution if no suitable resolution is found
        return max(resolutions.keys(), default=15)

    

#### Util GIS Functions

In [ ]:
import math
def square_degrees_to_sq_km(sq_degrees, latitude_deg=9.0):
    """
    Converts an area in square degrees to square kilometers.

    This conversion is an approximation because the physical size of a degree
    of longitude varies with latitude. The function uses the provided latitude
    to calculate a more accurate conversion factor. If no latitude is given,
    it defaults to the equator (0 degrees latitude).

    Args:
        sq_degrees (float): The area in square degrees.
        latitude_deg (float, optional): The latitude in degrees (e.g., 0 for equator).
                                        Defaults to 9.0 - Nigeria.
                                        Defaults to 0.0.

    Returns:
        float: The approximate area in square kilometers.
    """
    # Earth's mean radius in kilometers
    R_earth = 6371.0  # km

    # Convert latitude from degrees to radians
    latitude_rad = math.radians(latitude_deg)

    # Approximate length of one degree of latitude (constant)
    # This is roughly 111.132 km
    length_deg_lat_km = (2 * math.pi * R_earth) / 360

    # Approximate length of one degree of longitude at the given latitude
    # This varies with cosine of latitude
    length_deg_lon_km = length_deg_lat_km * math.cos(latitude_rad)

    # Area of one square degree at the given latitude
    area_1_sq_deg_at_lat_km2 = length_deg_lat_km * length_deg_lon_km

    # Convert the given square degrees to square kilometers
    area_km2 = sq_degrees * area_1_sq_deg_at_lat_km2

    return area_km2


# # Example usage:

# # Your value at the equator (latitude 0)
# sq_degrees_val = 0.000129
# area_in_km = square_degrees_to_sq_km(sq_degrees_val, latitude_deg=0.0)
# print(f"{sq_degrees_val} square degrees at the equator is approximately {area_in_km:.4f} km^2")

# # Example for a location in Nigeria (e.g., Abuja, approx 9 degrees North latitude)
# sq_degrees_val_nigeria = 0.000129
# latitude_nigeria = 9.0  # Degrees North
# area_in_km_nigeria = square_degrees_to_sq_km(sq_degrees_val_nigeria, latitude_nigeria)
# print(f"{sq_degrees_val_nigeria} square degrees at {latitude_nigeria}°N is approximately {area_in_km_nigeria:.4f} km^2")

# # Example for a higher latitude (e.g., London, approx 51.5 degrees North latitude)
# sq_degrees_val_london = 0.000129
# latitude_london = 51.5 # Degrees North
# area_in_km_london = square_degrees_to_sq_km(sq_degrees_val_london, latitude_london)
# print(f"{sq_degrees_val_london} square degrees at {latitude_london}°N is approximately {area_in_km_london:.4f} km^2")

# # Example of a larger area
# larger_sq_degrees = 1.0
# area_larger_at_equator = square_degrees_to_sq_km(larger_sq_degrees)
# print(f"\n{larger_sq_degrees} square degree at the equator is approximately {area_larger_at_equator:.4f} km^2")


#### Choosing Resolution

In [ ]:
H3Processor = H3ProcessorV4()
 
# # Examples
print(f"City analysis (100 km²): Resolution {H3Processor.recommend_resolution(100, 'analysis')}")
print(f"City analysis (100 km²): Resolution {H3Processor.recommend_resolution(100, 'visualization')}")
print(f"City analysis (100 km²): Resolution {H3Processor.recommend_resolution(100, 'overview')}")
print(f"Neighborhood analysis (1 km²): Resolution {H3Processor.recommend_resolution(1)}")
print(f"Building analysis (0.01 km²): Resolution {H3Processor.recommend_resolution(0.01)}")

# Display resolution characteristics
print("\nResolution characteristics:")
for res in range(5,16):
    area = h3.average_hexagon_area(res, 'km^2')
    edge = h3.average_hexagon_edge_length(res, 'km')
    num_cells = h3.get_num_cells(res)
    print(f"Resolution {res}: Area = {area:.6f} km², Edge = {edge:.3f} km , Cells  = {num_cells:,} total cells")


In [ ]:
# Example usage
min_radius_km = 5
max_radius_km = 9
recommended_resolution = H3Processor.recommend_resolution_for_radius(min_radius_km, max_radius_km, verbose =True)

min_radius_km_2 = 0.1
max_radius_km_2 = 0.2
recommended_resolution_2 = H3Processor.recommend_resolution_for_radius(min_radius_km_2, max_radius_km_2, verbose =True)

min_radius_km_3 = 1000
max_radius_km_3 = 2000
recommended_resolution_3 = H3Processor.recommend_resolution_for_radius(min_radius_km_3, max_radius_km_3, verbose =True)


#### Plotting Polygon

In [ ]:
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip
from shapely.geometry import Polygon # Used for dummy data

def add_ward_polygon_to_map(row_index, gdf, base_map=None, 
                            tooltip_fields=None, 
                            tooltip_aliases=None, add_tooltip=True,
                            fill_property = {'fillColor': 'blue', 'color': 'red', 'weight': 1, 'fillOpacity': 0.2}
                            ):
    """
    Adds a ward polygon to a Folium map for a given row of a GeoDataFrame.
    It returns the modified map object.

    Parameters:
    - row_index (int): The index of the row in the GeoDataFrame to plot.
    - gdf (geopandas.GeoDataFrame): The GeoDataFrame containing the geographic data.
    - base_map (folium.Map, optional): An existing Folium map object to use as the base map.
                                      If None, a new map will be created centered on the polygon.
    - tooltip_fields (list, optional): List of field names from the GeoDataFrame's properties
                                       to display in the tooltip. Required if add_tooltip is True.
    - tooltip_aliases (list, optional): List of aliases (labels) corresponding to the field names.
                                        Required if add_tooltip is True.
    - add_tooltip (bool, optional): Whether to add a tooltip to the polygon. Defaults to True.

    Returns:
    - folium.Map: The Folium map object with the added polygon.
    """
    if not 0 <= row_index < len(gdf):
        raise IndexError(f"Row index {row_index} is out of bounds for GeoDataFrame with {len(gdf)} rows.")

    # Extract the specified row as a GeoJSON Feature
    feature_geojson = gdf.iloc[[row_index]].__geo_interface__['features'][0]

    # Create or use the provided Folium map
    if base_map is None:
        # Get the geometry for centroid calculation
        geometry = gdf.iloc[row_index]['geometry']
        
        # Access the CRS from the GeoDataFrame itself
        current_gdf_crs = gdf.crs

        if current_gdf_crs and current_gdf_crs != 'EPSG:4326':
            centroid_gs = gpd.GeoSeries([geometry.centroid], crs=current_gdf_crs)
            centroid_4326 = centroid_gs.to_crs(epsg=4326).iloc[0]
            map_center = [centroid_4326.y, centroid_4326.x]
        else:
            centroid = geometry.centroid
            map_center = [centroid.y, centroid.x]
            
        m = folium.Map(location=map_center, zoom_start=12)
    else:
        m = base_map

    # Define styling for the polygon (you could make this dynamic based on row properties too!)
    style_function = lambda x: {
        'fillColor': fill_property.get('fillColor', 'blue'),
        'color': fill_property.get('color', 'red'), 
        'weight': fill_property.get('weight', 1),  
        'fillOpacity': fill_property.get('fillOpacity', 0.1) 
    }

    # Safely get ward name for the layer control
    ward_name = feature_geojson['properties'].get('wardname', f"Ward {row_index}")

    # Conditionally create the tooltip object
    feature_tooltip = None
    if add_tooltip and tooltip_fields and tooltip_aliases:
        if len(tooltip_fields) != len(tooltip_aliases):
            print("Warning: tooltip_fields and tooltip_aliases lists have different lengths. Tooltip may not display correctly.")
        feature_tooltip = GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            # localize=True,
            sticky=True
        )
    elif add_tooltip:
        print("Warning: add_tooltip is True but tooltip_fields or tooltip_aliases are missing. Tooltip will not be added.")

    # Add the GeoJSON to the map
    folium.GeoJson(
        feature_geojson,
        style_function=style_function,
        name=ward_name,
        tooltip=feature_tooltip
    ).add_to(m)

    return m

##### Testing

In [ ]:
# --- Dummy Data Setup (from your previous request) ---
# Define three distinct dummy polygons
dummy_polygon1 = Polygon([(3.3, 6.5), (3.3, 6.6), (3.4, 6.6), (3.4, 6.5)])
dummy_polygon2 = Polygon([(3.31, 6.51), (3.31, 6.61), (3.401, 6.61), (3.41, 6.51)])
dummy_polygon3 = Polygon([(3.5, 6.7), (3.5, 6.8), (3.6, 6.8), (3.6, 6.7)])

# Create the dictionary for the GeoDataFrame
dummy_data = {
    'wardname': ['Lagos Island East', 'Eti Osa West', 'Apapa Wharf'],
    'wardcode': ['LI001', 'EO002', 'AW003'],
    'lganame': ['Lagos Island', 'Eti Osa', 'Apapa'],
    'lgacode': ['LIA001', 'EOA002', 'APA003'],
    'statename': ['Lagos', 'Lagos', 'Lagos'],
    'statecode': ['LAG', 'LAG', 'LAG'],
    'amapcode': ['AMAPLAG001', 'AMAPLAG002', 'AMAPLAG003'],
    'status': ['Active', 'Active', 'Pending'],
    'source': ['Dummy', 'Dummy', 'Generated'],
    'urban': ['Yes', 'Yes', 'No'],
    'shape__area': [dummy_polygon1.area, dummy_polygon2.area, dummy_polygon3.area],
    'shape__length': [dummy_polygon1.length, dummy_polygon2.length, dummy_polygon3.length],
    'geometry': [dummy_polygon1, dummy_polygon2, dummy_polygon3]
}

gdf = gpd.GeoDataFrame(dummy_data, crs="EPSG:4326")

# Add a calculated 'area_sq_km' column to the gdf for the tooltip example
# Reproject to an equal-area CRS (e.g., EPSG:6933) for calculation
gdf_proj_for_area = gdf.to_crs(epsg=6933) # WGS 84 / World Cylindrical Equal Area
gdf['area_sq_km'] = gdf_proj_for_area.geometry.area / 1_000_000 # Convert m^2 to km^2


# --- Main script to plot multiple polygons ---

# 1. Initialize the base map once 
all_polygons_union = gdf.geometry.unary_union
overall_centroid = all_polygons_union.centroid

# Reproject the overall centroid to EPSG:4326 for Folium if necessary
current_gdf_crs = gdf.crs
if current_gdf_crs and current_gdf_crs != 'EPSG:4326':
    centroid_gs = gpd.GeoSeries([overall_centroid], crs=current_gdf_crs)
    centroid_4326 = centroid_gs.to_crs(epsg=4326).iloc[0]
    initial_map_center = [centroid_4326.y, centroid_4236.x]
else:
    initial_map_center = [overall_centroid.y, overall_centroid.x]
    
m = folium.Map(location=initial_map_center, zoom_start=10)

tooltip_fields = ['wardname', 'lganame', 'urban', 'area_sq_km']
tooltip_aliases = ['Ward Name:', 'LGA Name:', 'Urban Status:', 'Area (km²):']
output_map_path = "three_wards_map.html"

# 2. Loop through all rows in your GeoDataFrame
for i in range(len(gdf)):
    # Add each polygon to the *same* map object 'm'
    m = add_ward_polygon_to_map(
        i, # Current row index
        gdf,
        base_map=m, # Pass the existing map object
        tooltip_fields=tooltip_fields,
        tooltip_aliases=tooltip_aliases,
        add_tooltip=True
    )

# 3. Add Layer Control once, after all polygons have been added
folium.LayerControl().add_to(m)

# 4. Save the final map once
m.save(output_map_path)

print(f"Map with three polygons saved to {output_map_path}")

# In a Jupyter Notebook, you can just run 'm' to display it
m

##### Plotting Lagos Ward

In [ ]:
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip
import pickle



def create_ward_map(gdf, style_function_ = None):
    
    if style_function_ is None:
        def style_function(x):
            return {'fillColor': '#3E07BD', 'color': '#6D6B6B', 'weight': 0.7, 'fillOpacity': 0.2}
        return style_function
        
    # Project the GeoDataFrame to a suitable CRS for area calculation
    gdf_proj = gdf.to_crs("EPSG:6933")  # WGS 84 / World Cylindrical Equal Area
    gdf['area_sq_km'] = (gdf_proj.geometry.area / 1_000_000).round(4)  # Convert m^2 to km^2

    # Calculate the centroid of the entire geometry
    all_polygons_union = gdf.geometry.union_all()
    overall_centroid = all_polygons_union.centroid

    # Reproject the overall centroid to EPSG:4326 for Folium if necessary
    current_gdf_crs = gdf.crs
    if current_gdf_crs and current_gdf_crs != 'EPSG:4326':
        centroid_gs = gpd.GeoSeries([overall_centroid], crs=current_gdf_crs)
        centroid_4326 = centroid_gs.to_crs(epsg=4326).iloc[0]
        initial_map_center = [centroid_4326.y, centroid_4326.x]
    else:
        initial_map_center = [overall_centroid.y, overall_centroid.x]

    # Initialize the base map
    map_tiles = {
        'CartoDB Positron': "CartoDB Positron",
        'CartoDB Voyager': "CartoDB Voyager",
        'OpenStreetMap': 'OpenStreetMap',
    }
    map_lagos_wards = folium.Map(location=initial_map_center, zoom_start=11, tiles=map_tiles['CartoDB Voyager'])

    # Define tooltip fields and aliases
    tooltip_fields = ['wardname', 'lganame', 'urban', 'area_sq_km']
    tooltip_aliases = ['Ward Name:', 'LGA Name:', 'Urban Status:', 'Area (km²):']

    # Loop through all rows in your GeoDataFrame and add polygons to the map
    for i in range(len(gdf)):
        folium.GeoJson(
            gdf.iloc[i:i+1],
            tooltip=folium.features.GeoJsonTooltip(
                fields=tooltip_fields,
                aliases=tooltip_aliases,
                localize=True
            ),
            style_function=style_function_
        ).add_to(map_lagos_wards)

    # Return the map object
    return map_lagos_wards

# Example usage:
def style_function(x):
    return {'fillColor': '#3E07BD', 'color': '#6D6B6B', 'weight': 0.7, 'fillOpacity': 0.2}

map_lagos_wards = create_ward_map(dict_causeway['df_ng_ward_lagos'], style_function_=style_function)

# Save the map object to a file
with open('./input/map_lagos_wards.pkl', 'wb') as f:
    pickle.dump(map_lagos_wards, f)


##### Convert square degrees to sq km

In [ ]:

# # Usage example
# processor = H3ProcessorV4(resolution=9)

# # Sample data
# data = pd.DataFrame({
#     'lat': [37.7749, 37.7849, 37.7649],
#     'lng': [-122.4194, -122.4094, -122.4294],
#     'sales': [100, 200, 150]
# })

# # Process data
# processed = processor.process_points(data)
# aggregated = processor.aggregate_by_h3(processed, 'sales')
# print("Aggregated data:")
# print(aggregated)

# # Get detailed cell information
# sample_cell = aggregated.iloc[0]['h3_index']
# cell_info = processor.get_cell_info(sample_cell)
# print(f"\nCell information for {sample_cell}:")
# for key, value in cell_info.items():
#     print(f"  {key}: {value}")

# # Get neighbors information
# neighbors_info = processor.get_neighbors_info(sample_cell, k=1)
# print(f"\nNeighbors information:")
# for neighbor in neighbors_info[:4]:  # Show first 4
#     print(f"  {neighbor['h3_index']}: {neighbor['distance_km']:.3f}km away")


## H3 Clustering

In [ ]:
def clean_df_coordinates(df, lat_col='lat', lon_col='lon', return_point = False):
    """
    Convert a pandas DataFrame to a list of (lat, lon) tuples.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing latitude and longitude columns
    lat_col : str, default 'lat'
        Name of the latitude column
    lon_col : str, default 'lon'
        Name of the longitude column
    
    Returns:
    --------
    list
        List of (lat, lon) tuples
    
    Raises:
    -------
    ValueError
        If specified columns don't exist in the DataFrame
    KeyError
        If DataFrame is empty or columns contain invalid data
    
    Examples:
    ---------
    >>> df = pd.DataFrame({
    ...     'latitude': [40.7128, 34.0522, 41.8781],
    ...     'longitude': [-74.0060, -118.2437, -87.6298]
    ... })
    >>> points = dataframe_to_points(df, lat_col='latitude', lon_col='longitude')
    >>> print(points)
    [(40.7128, -74.0060), (34.0522, -118.2437), (41.8781, -87.6298)]
    """
    # Validate inputs
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame")
    
    if df.empty:
        raise ValueError("DataFrame is empty")
    
    if lat_col not in df.columns:
        raise ValueError(f"Column '{lat_col}' not found in DataFrame. Available columns: {list(df.columns)}")
    
    if lon_col not in df.columns:
        raise ValueError(f"Column '{lon_col}' not found in DataFrame. Available columns: {list(df.columns)}")
    
    # Convert the lat and long to numeric columns
    df[[lat_col, lon_col]] = df[[lat_col, lon_col]].apply(pd.to_numeric, errors='coerce')
    
    
    # Check for null values
    null_lat = df[lat_col].isnull().sum()
    null_lon = df[lon_col].isnull().sum()
    
    if null_lat > 0 or null_lon > 0:
        print(f"Warning: Found {null_lat} null values in {lat_col} and {null_lon} null values in {lon_col}")
        print("Dropping rows with null coordinates...")
        df = df.dropna(subset=[lat_col, lon_col])
    
    # Validate coordinate ranges
    invalid_lat = df[(df[lat_col] < -90) | (df[lat_col] > 90)]
    invalid_lon = df[(df[lon_col] < -180) | (df[lon_col] > 180)]
    
    if not invalid_lat.empty:
        print(f"Warning: Found {len(invalid_lat)} rows with invalid latitude values (outside -90 to 90)")
    
    if not invalid_lon.empty:
        print(f"Warning: Found {len(invalid_lon)} rows with invalid longitude values (outside -180 to 180)")
    
    # Filter out invalid coordinates
    df_clean = df[
        (df[lat_col] >= -90) & (df[lat_col] <= 90) &
        (df[lon_col] >= -180) & (df[lon_col] <= 180)
    ]
    
    if df_clean.empty:
        raise ValueError("No valid coordinates found after filtering")
    
    if return_point:
        # Convert to list of tuples
        points = list(zip(df_clean[lat_col], df_clean[lon_col]))
    
        print(f"Successfully converted {len(points)} valid coordinates to points")
        return points
    else:
        print(f"Dataframe now contains {len(df_clean):,} valid coordinates")
        return df_clean

In [ ]:
df_clean = clean_df_coordinates(df = dict_causeway['sp_active_customers_dim'], lat_col='Latitude', lon_col='Longitude', return_point = False)
df_clean.shape

In [ ]:
import h3
import folium
import pandas as pd
from folium.plugins import FastMarkerCluster, HeatMap
from shapely.geometry import Polygon
import random

class H3MapVisualizer_deprecated:
    def __init__(self, resolution=7, zoom_start=4):
        self.resolution = resolution
        self.zoom_start = zoom_start
        self.hex_counts = None
        self.df = None
        
    def generate_sample_data(self):
        """Generate sample data with 25+ points across major US cities"""
        cities = {
            'NYC': [(40.7128, -74.0060), (40.7210, -74.0005), (40.7050, -74.0100), 
                   (40.7282, -73.9942), (40.7505, -73.9934), (40.7614, -73.9776)],
            'LA': [(34.0522, -118.2437), (34.0689, -118.2578), (34.0407, -118.2468),
                  (34.0928, -118.3287), (34.0195, -118.4912)],
            'Chicago': [(41.8781, -87.6298), (41.8900, -87.6200), (41.8675, -87.6188),
                       (41.8994, -87.6347), (41.8369, -87.6847)],
            'Miami': [(25.7617, -80.1918), (25.7907, -80.1300), (25.7480, -80.2078),
                     (25.7825, -80.2994)],
            'Seattle': [(47.6062, -122.3321), (47.6205, -122.3493), (47.5990, -122.3350),
                       (47.6097, -122.3331)],
            'Denver': [(39.7392, -104.9903), (39.7817, -104.9667), (39.7294, -105.0178)],
            'Austin': [(30.2672, -97.7431), (30.3072, -97.7559), (30.2500, -97.7500)],
            'Boston': [(42.3601, -71.0589), (42.3736, -71.0275), (42.3505, -71.0709)]
        }
        
        points = []
        for city_points in cities.values():
            points.extend(city_points)
            
        # Add some random variations around existing points
        additional_points = []
        for lat, lon in points[:10]:  # Add variations to first 10 points
            additional_points.append((
                lat + random.uniform(-0.01, 0.01),
                lon + random.uniform(-0.01, 0.01)
            ))
        
        points.extend(additional_points)
        return points
    
    def process_data(self, points):
        """Process points data and generate H3 hexagon counts"""
        self.df = pd.DataFrame(points, columns=['lat', 'lon'])
        self.df['hex_id'] = self.df.apply(
            lambda row: h3.latlng_to_cell(row['lat'], row['lon'], self.resolution), 
            axis=1
        )
        
        # Aggregate counts per hexagon
        self.hex_counts = self.df.groupby('hex_id').size().reset_index(name='point_count')
        
        print(f"Generated {len(points)} points")
        print(f"Created {len(self.hex_counts)} hexagons")
        print(f"Max points per hexagon: {self.hex_counts['point_count'].max()}")
        
        return self.hex_counts
    
    def create_base_map(self, points):
        """Create the base folium map"""
        center_lat = sum(p[0] for p in points) / len(points)
        center_lon = sum(p[1] for p in points) / len(points)
        
        m = folium.Map(
            location=[center_lat, center_lon],
            tiles='cartodbpositron',
            zoom_start=self.zoom_start
        )
        
        # Add title with horizontal legend as navbar
        title_html = '''
        <div style="position: fixed; 
                    top: 10px; left: 50%; transform: translateX(-50%); 
                    background-color: rgba(255, 255, 255, 0.95); 
                    padding: 15px 25px; border-radius: 10px; 
                    box-shadow: 0 2px 10px rgba(0,0,0,0.1); 
                    z-index: 1000; font-family: Arial, sans-serif;
                    border: 1px solid #ddd;">
            <h3 style="margin: 0 0 10px 0; color: #2c3e50; font-size: 18px; text-align: center;">
                H3 Grid Analysis Dashboard
            </h3>
            <div style="display: flex; justify-content: center; align-items: center; gap: 20px; font-size: 13px;">
                <span style="font-weight: bold; color: #34495e;">Legend:</span>
                <div style="display: flex; align-items: center; gap: 5px;">
                    <div style="width: 16px; height: 16px; background-color: #3388ff; opacity: 0.7; border-radius: 3px;"></div>
                    <span>1 Point</span>
                </div>
                <div style="display: flex; align-items: center; gap: 5px;">
                    <div style="width: 16px; height: 16px; background-color: #ff7800; opacity: 0.7; border-radius: 3px;"></div>
                    <span>2-3 Points</span>
                </div>
                <div style="display: flex; align-items: center; gap: 5px;">
                    <div style="width: 16px; height: 16px; background-color: #d63031; opacity: 0.7; border-radius: 3px;"></div>
                    <span>4+ Points</span>
                </div>
            </div>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(title_html))
        
        return m
    
    def add_heatmap_layer(self, map_obj, points):
        """Add heatmap layer to the map"""
        HeatMap(
            data=points,
            radius=15,
            blur=10,
            min_opacity=0.3,
            gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}
        ).add_to(folium.FeatureGroup(name='Heatmap').add_to(map_obj))
        
        return map_obj
    
    def get_hex_color(self, count):
        """Get color based on point count"""
        if count == 10:
            return '#3388ff'  # Blue for single points
        elif count <= 20:
            return '#ff7800'  # Orange for moderate density
        else:
            return '#d63031'  # Red for high density
    
    def add_hexagon_layer(self, map_obj, hex_data):
        """Add H3 hexagon layer with proper styling"""
        hex_group = folium.FeatureGroup(name='H3 Hexagons')
        
        for _, row in hex_data.iterrows():
            try:
                # Get hexagon boundary - returns list of (lat, lon) tuples
                hex_boundary = h3.cell_to_boundary(row['hex_id'])
                
                # Convert to (lon, lat) for GeoJSON format
                hex_coords = [(lon, lat) for lat, lon in hex_boundary]
                
                # Create GeoJSON polygon directly
                hex_geojson = {
                    "type": "Feature",
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [hex_coords]
                    },
                    "properties": {
                        "hex_id": row['hex_id'],
                        "point_count": row['point_count']
                    }
                }
                
                # Determine color based on point count
                fill_color = self.get_hex_color(row['point_count'])
                
                # Add hexagon to map
                folium.GeoJson(
                    hex_geojson,
                    style_function=lambda feature, color=fill_color: {
                        'fillColor': color,
                        'color': 'black',
                        'weight': 2,
                        'fillOpacity': 0.7,
                        'opacity': 1.0
                    },
                    tooltip=folium.Tooltip(
                        f"<b>Hex ID:</b> {row['hex_id']}<br>"
                        f"<b>Points:</b> {row['point_count']}<br>"
                        f"<b>Resolution:</b> {self.resolution}",
                        sticky=True
                    )
                ).add_to(hex_group)
                
            except Exception as e:
                print(f"Error processing hex {row['hex_id']}: {e}")
                continue
        
        hex_group.add_to(map_obj)
        print(f"Added {len(hex_data)} hexagons to map")
        return map_obj
    
    def add_marker_cluster(self, map_obj, points):
        """Add clustered markers for individual points"""
        FastMarkerCluster(
            data=[[p[0], p[1]] for p in points],
            name='Point Clusters',
            disable_clustering_at_zoom=12,
            show_coverage_on_hover=False,
            icon_create_function="""
            function(cluster) {
                var count = cluster.getChildCount();
                var size = count < 10 ? 'small' : count < 50 ? 'medium' : 'large';
                return L.divIcon({
                    html: '<b>' + count + '</b>',
                    className: 'marker-cluster marker-cluster-' + size,
                    iconSize: new L.Point(30, 30)
                });
            }
            """
        ).add_to(map_obj)
        
        return map_obj
    
    def add_legend(self, map_obj):
        """Add a legend to explain the color coding"""
        legend_html = '''
        <div style="position: fixed; 
                    top: 10px; right: 50px; width: 200px; height: 120px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <h4 style="margin-top:0">Hexagon Legend</h4>
        <i style="background:#3388ff; width:20px; height:20px; float:left; margin-right:8px; opacity:0.7"></i>1 Point<br>
        <i style="background:#ff7800; width:20px; height:20px; float:left; margin-right:8px; opacity:0.7"></i>2-3 Points<br>
        <i style="background:#d63031; width:20px; height:20px; float:left; margin-right:8px; opacity:0.7"></i>4+ Points<br>
        </div>
        '''
        map_obj.get_root().html.add_child(folium.Element(legend_html))
        return map_obj
    
    def debug_hexagons(self, hex_data, num_to_show=3):
        """Debug method to print hexagon coordinates"""
        print(f"\nDEBUG: Showing first {num_to_show} hexagons:")
        for i, (_, row) in enumerate(hex_data.head(num_to_show).iterrows()):
            print(f"\nHexagon {i+1}:")
            print(f"  Hex ID: {row['hex_id']}")
            print(f"  Point count: {row['point_count']}")
            
            try:
                boundary = h3.cell_to_boundary(row['hex_id'])
                print(f"  Boundary coordinates: {boundary}")
                
                # Check if coordinates are reasonable
                lats = [coord[0] for coord in boundary]
                lons = [coord[1] for coord in boundary]
                print(f"  Lat range: {min(lats):.4f} to {max(lats):.4f}")
                print(f"  Lon range: {min(lons):.4f} to {max(lons):.4f}")
                
            except Exception as e:
                print(f"  Error getting boundary: {e}")
    
    def create_visualization(self, points=None):
        """Main method to create the complete visualization"""
        if points is None:
            points = self.generate_sample_data()
        
        # Process data
        hex_counts = self.process_data(points)
        
        # Debug hexagons
        # self.debug_hexagons(hex_counts)
        
        # Create base map
        m = self.create_base_map(points)
        
        # Add layers
        m = self.add_heatmap_layer(m, points)
        m = self.add_hexagon_layer(m, hex_counts)
        m = self.add_marker_cluster(m, points)
        
        # Add layer control (removed separate legend since it's now in the title)
        folium.LayerControl(position='topright', collapsed=False).add_to(m)
        
        return m


In [ ]:
import h3
import folium
import pandas as pd
from folium.plugins import FastMarkerCluster, HeatMap
import random
from typing import Optional, Union, List, Dict, Tuple, Any

class H3MapVisualizer:
    """
    A versatile class for visualizing H3 geospatial data using Folium.

    Supports different coloring schemes for H3 hexagons and includes
    optional heatmap and marker cluster layers.
    """

    def __init__(
        self,
        resolution: int = 7,
        zoom_start: int = 4,
        color_scheme: Optional[Union[str, Dict]] = None
    ):
        """
        Initialize H3 map visualizer with flexible coloring schemes.

        Args:
            resolution: H3 resolution level (1-15). Lower numbers mean larger hexagons.
            zoom_start: Initial map zoom level.
            color_scheme: Defines how hexagons are colored. Can be:
                - None or 'default': Uses a predefined threshold-based scheme.
                - 'percentage': Uses a percentage-based coloring scheme (low, medium, high).
                - Dict: A custom scheme configuration. Expected keys vary by 'type':
                    - {'type': 'thresholds', 'thresholds': List[int], 'colors': List[str], 'labels': List[str]}
                    - {'type': 'percentage', 'colors': List[str], 'labels': List[str]}
        """
        if not 1 <= resolution <= 15:
            raise ValueError("H3 resolution must be between 1 and 15.")
        if not 1 <= zoom_start <= 18: # Reasonable max zoom for most map tiles
            raise ValueError("Zoom start must be between 1 and 18.")

        self.resolution = resolution
        self.zoom_start = zoom_start
        self.df: Optional[pd.DataFrame] = None
        self.hex_counts: Optional[pd.DataFrame] = None
        self.min_count: int = 0
        self.max_count: int = 1 # Initialize to 1 to avoid ZeroDivisionError if no points

        # Initialize color scheme based on input
        self._init_color_scheme(color_scheme)

    def _init_color_scheme(self, color_scheme_input: Optional[Union[str, Dict]]) -> None:
        """
        Initializes and validates the color scheme configuration.
        """
        default_threshold_scheme = {
            'type': 'thresholds',
            'thresholds': [1, 5, 10, 25, 50], # More granular default thresholds
            'colors': ['#e0f2f7', '#a7d9ed', '#6ebcdb', '#359ec9', '#0070a3', '#004060'],
            'labels': ['1', '2-5', '6-10', '11-25', '26-50', '50+']
        }
        default_percentage_scheme = {
            'type': 'percentage',
            'colors': ['#e0f7fa', '#80deea', '#00bcd4', '#00838f'], # Adjusted colors for percentage
            'labels': ['Lowest 25%', '25%-50%', '50%-75%', 'Highest 25%']
        }

        if color_scheme_input is None or (isinstance(color_scheme_input, str) and color_scheme_input.lower() == 'default'):
            self.color_scheme = default_threshold_scheme
        elif isinstance(color_scheme_input, str) and color_scheme_input.lower() == 'percentage':
            self.color_scheme = default_percentage_scheme
        elif isinstance(color_scheme_input, Dict):
            # Validate custom dictionary scheme
            scheme_type = color_scheme_input.get('type')
            if scheme_type == 'thresholds':
                if not all(k in color_scheme_input for k in ['thresholds', 'colors']):
                    raise ValueError("Custom 'thresholds' color scheme must include 'thresholds' and 'colors'.")
                if len(color_scheme_input['colors']) != len(color_scheme_input['thresholds']) + 1:
                     raise ValueError("For 'thresholds' scheme, 'colors' list must be one element longer than 'thresholds'.")
                if not all(isinstance(t, int) and t >= 0 for t in color_scheme_input['thresholds']):
                    raise ValueError("Thresholds must be non-negative integers.")
                self.color_scheme = color_scheme_input
            elif scheme_type == 'percentage':
                if not all(k in color_scheme_input for k in ['colors', 'labels']):
                    raise ValueError("Custom 'percentage' color scheme must include 'colors' and 'labels'.")
                self.color_scheme = color_scheme_input
            else:
                raise ValueError(f"Unknown custom color scheme type: {scheme_type}. Supported types: 'thresholds', 'percentage'.")
        else:
            raise ValueError("Invalid color_scheme type. Must be None, 'percentage', 'default', or a dictionary.")

    def get_hex_color(self, count: int) -> str:
        """
        Determines the appropriate color for a hexagon based on its point count
        and the configured color scheme.

        Args:
            count: The number of points within the hexagon.

        Returns:
            A hex color code (e.g., '#RRGGBB').
        """
        scheme_type = self.color_scheme['type']
        colors = self.color_scheme['colors']

        if scheme_type == 'thresholds':
            thresholds = self.color_scheme['thresholds']
            for i, threshold in enumerate(thresholds):
                if count <= threshold:
                    return colors[i]
            return colors[-1] # Fallback for counts exceeding all thresholds

        elif scheme_type == 'percentage':
            if self.max_count <= 1:
                return colors[0] # Default for single or zero points

            percentage = count / self.max_count
            if percentage <= 0.25:
                return colors[0]
            elif percentage <= 0.50:
                return colors[1]
            elif percentage <= 0.75:
                return colors[2]
            else:
                return colors[3] # For the top 25%

        # This should ideally not be reached due to validation in _init_color_scheme
        return '#808080' # Grey fallback

    def add_legend(self, map_obj: folium.Map, title: str = "H3 Grid Analysis") -> folium.Map:
        """
        Adds a responsive and informative legend to the map based on the
        configured color scheme.

        Args:
            map_obj: The Folium map instance to which the legend will be added.
            title: The title to display above the legend.

        Returns:
            The Folium map instance with the added legend.
        """
        scheme_type = self.color_scheme['type']
        items: List[str]

        if scheme_type == 'thresholds':
            items = self._create_threshold_legend_items()
        elif scheme_type == 'percentage':
            items = self._create_percentage_legend_items()
        else:
            # Fallback for unforeseen scheme types (though validation should prevent this)
            items = []

        legend_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
                    background: white; padding: 10px 15px; border-radius: 8px;
                    box-shadow: 0 2px 10px rgba(0,0,0,0.1); z-index: 1000;
                    font-family: Arial, sans-serif; max-width: 90%; overflow-x: auto;">
            <h3 style="margin: 0 0 8px 0; text-align: center; font-size: 16px;">{title}</h3>
            <div style="display: flex; flex-wrap: wrap; justify-content: center; gap: 10px;
                        font-size: 12px;">
                {''.join(items)}
            </div>
        </div>
        '''
        map_obj.get_root().html.add_child(folium.Element(legend_html))
        return map_obj

    def _create_threshold_legend_items(self) -> List[str]:
        """
        Generates HTML legend items for a threshold-based coloring scheme.
        """
        items = []
        thresholds = self.color_scheme['thresholds']
        colors = self.color_scheme['colors']
        labels = self.color_scheme.get('labels')

        for i in range(len(colors)):
            label: str
            if labels and i < len(labels):
                label = labels[i]
            elif i == 0:
                label = f"1-{thresholds[0]}"
            elif i < len(thresholds):
                label = f"{thresholds[i-1] + 1}-{thresholds[i]}"
            else: # Last item for "above max threshold"
                label = f"{thresholds[-1] + 1}+"

            items.append(f'''
                <div style="display: flex; align-items: center; margin: 0 10px;">
                    <div style="width: 20px; height: 20px; background: {colors[i]};
                                 border: 1px solid #333; margin-right: 5px;"></div>
                    <span>{label}</span>
                </div>
            ''')
        return items

    def _create_percentage_legend_items(self) -> List[str]:
        """
        Generates HTML legend items for a percentage-based coloring scheme.
        """
        items = []
        colors = self.color_scheme['colors']
        labels = self.color_scheme.get('labels', ['Lowest', 'Low-Medium', 'Medium-High', 'Highest'])

        # Calculate approximate ranges for display in legend if actual ranges aren't provided
        # This assumes even distribution for simplicity, for actual ranges, labels should be explicit
        if self.max_count > 1:
            # Using 4 categories for percentage scheme
            ranges = [
                f"1-{int(self.max_count * 0.25)}",
                f"{int(self.max_count * 0.25) + 1}-{int(self.max_count * 0.50)}",
                f"{int(self.max_count * 0.50) + 1}-{int(self.max_count * 0.75)}",
                f"{int(self.max_count * 0.75) + 1}-{self.max_count}"
            ]
        else: # Handle case with very few points or max_count of 1
            ranges = ["N/A", "N/A", "N/A", "N/A"]

        for i, color in enumerate(colors):
            label = labels[i] if i < len(labels) else "N/A"
            current_range = ranges[i] if i < len(ranges) else "N/A"
            items.append(f'''
                <div style="display: flex; align-items: center; margin: 0 10px;">
                    <div style="width: 20px; height: 20px; background: {color};
                                 border: 1px solid #333; margin-right: 5px;"></div>
                    <span>{label} ({current_range})</span>
                </div>
            ''')
        return items

    def create_base_map(self, points: List[Tuple[float, float]]) -> folium.Map:
        """
        Creates a base Folium map centered on the provided points.

        Args:
            points: A list of (latitude, longitude) tuples.

        Returns:
            A Folium map instance.
        """
        if not points:
            # Default to a general US center or allow user to specify a fallback center
            center_lat, center_lon = 39.8283, -98.5795 # Geographic center of the contiguous US
            print("Warning: No points provided to center map. Defaulting to continental US center.")
        else:
            center_lat = sum(p[0] for p in points) / len(points)
            center_lon = sum(p[1] for p in points) / len(points)

        return folium.Map(
            location=[center_lat, center_lon],
            tiles='cartodbpositron',
            zoom_start=self.zoom_start
        )

    def add_heatmap_layer(self, map_obj: folium.Map, points: List[Tuple[float, float]]) -> folium.Map:
        """
        Adds a heatmap layer to the map.

        Args:
            map_obj: The Folium map instance.
            points: A list of (latitude, longitude) tuples for the heatmap.

        Returns:
            The Folium map instance with the added heatmap.
        """
        if not points:
            print("Warning: No points provided for heatmap layer. Skipping.")
            return map_obj

        HeatMap(
            data=points,
            radius=15,
            blur=10,
            min_opacity=0.3,
            gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}
        ).add_to(folium.FeatureGroup(name='Heatmap').add_to(map_obj))
        return map_obj

    def add_hexagon_layer(self, map_obj: folium.Map, hex_data: pd.DataFrame) -> folium.Map:
        """
        Adds H3 hexagon layer to the map, with coloring based on point density.

        Args:
            map_obj: The Folium map instance.
            hex_data: A Pandas DataFrame containing 'hex_id' and 'point_count'.

        Returns:
            The Folium map instance with the added hexagon layer.
        """
        if hex_data.empty:
            print("Warning: No H3 hexagon data to display. Skipping hexagon layer.")
            return map_obj

        hex_group = folium.FeatureGroup(name='H3 Hexagons')

        # Use apply instead of iterrows for potentially better performance on large DFs
        def style_function(feature: Dict[str, Any]) -> Dict[str, Any]:
            count = feature['properties']['point_count']
            return {
                'fillColor': self.get_hex_color(count),
                'color': 'black',
                'weight': 2,
                'fillOpacity': 0.7
            }

        def tooltip_function(feature: Dict[str, Any]) -> str:
            props = feature['properties']
            return (f"<b>Hex ID:</b> {props['hex_id']}<br>"
                    f"<b>Points:</b> {props['point_count']}<br>"
                    f"<b>Resolution:</b> {self.resolution}")

        for _, row in hex_data.iterrows():
            try:
                # h3.cell_to_boundary returns (lat, lon) pairs
                hex_boundary = h3.cell_to_boundary(row['hex_id'])
                # GeoJson expects (lon, lat)
                hex_coords = [(lon, lat) for lat, lon in hex_boundary]

                folium.GeoJson(
                    {
                        "type": "Feature",
                        "geometry": {
                            "type": "Polygon",
                            "coordinates": [hex_coords]
                        },
                        "properties": row.to_dict() # Pass all row properties
                    },
                    style_function=style_function,
                    tooltip=folium.Tooltip(tooltip_function(
                        {"properties": row.to_dict()} # Pass properties to tooltip function
                    ), sticky=True)
                ).add_to(hex_group)
            except Exception as e:
                print(f"Error processing hex {row['hex_id']}: {e}. Skipping this hexagon.")

        hex_group.add_to(map_obj)
        return map_obj

    def add_marker_cluster(self, map_obj: folium.Map, points: List[Tuple[float, float]]) -> folium.Map:
        """
        Add a marker cluster layer for individual points.

        Args:
            map_obj: The Folium map instance.
            points: A list of (latitude, longitude) tuples to cluster.

        Returns:
            The Folium map instance with the added marker cluster.
        """
        if not points:
            print("Warning: No points provided for marker cluster. Skipping.")
            return map_obj

        FastMarkerCluster(
            data=[[p[0], p[1]] for p in points],
            name='Point Clusters',
            disable_clustering_at_zoom=12,
            show_coverage_on_hover=False,
            icon_create_function=self._create_cluster_icon()
        ).add_to(map_obj)
        return map_obj

    def _create_cluster_icon(self) -> str:
        """
        Generates the JavaScript function for custom marker cluster icons.
        """
        return """
        function(cluster) {
            var count = cluster.getChildCount();
            var size = count < 10 ? 'small' : count < 50 ? 'medium' : 'large';
            return L.divIcon({
                html: '<b>' + count + '</b>',
                className: 'marker-cluster marker-cluster-' + size,
                iconSize: new L.Point(30, 30)
            });
        }
        """

    def generate_sample_data(self, num_points: int = 500) -> List[Tuple[float, float]]:
        """
        Generates sample geographic data points across multiple simulated "hotspots".

        Args:
            num_points: The total number of sample points to generate.

        Returns:
            A list of (latitude, longitude) tuples.
        """
        # Define a few "hotspot" centers
        hotspots = [
            (34.0522, -118.2437),  # Los Angeles
            (40.7128, -74.0060),   # New York City
            (41.8781, -87.6298),   # Chicago
            (29.7604, -95.3698),   # Houston
            (33.7490, -84.3880)    # Atlanta
        ]
        points = []
        for _ in range(num_points):
            # Randomly pick a hotspot
            center_lat, center_lon = random.choice(hotspots)
            # Add some random offset to simulate dispersion around the hotspot
            lat = center_lat + random.uniform(-0.1, 0.1)
            lon = center_lon + random.uniform(-0.1, 0.1)
            points.append((lat, lon))
        return points

    def process_data(self, points: List[Tuple[float, float]]) -> pd.DataFrame:
        """
        Processes raw geographic points into H3 hexagons and calculates point counts.

        This method updates the internal `hex_counts`, `min_count`, and `max_count`
        attributes, which are crucial for color assignment and legend generation.

        Args:
            points: A list of (latitude, longitude) tuples.

        Returns:
            A Pandas DataFrame with 'hex_id' and 'point_count' columns.
        """
        if not points:
            self.df = pd.DataFrame(columns=['lat', 'lon', 'hex_id'])
            self.hex_counts = pd.DataFrame(columns=['hex_id', 'point_count'])
            self.min_count = 0
            self.max_count = 1 # Keep at 1 to prevent division by zero in get_hex_color
            print("No points to process.")
            return self.hex_counts


        self.df = pd.DataFrame(points, columns=['lat', 'lon'])
        self.df['hex_id'] = self.df.apply(
            lambda row: h3.latlng_to_cell(row['lat'], row['lon'], self.resolution),
            axis=1
        )

        self.hex_counts = self.df.groupby('hex_id').size().reset_index(name='point_count')
        self.min_count = self.hex_counts['point_count'].min()
        self.max_count = self.hex_counts['point_count'].max()

        print(f"Processed {len(points)} points into {len(self.hex_counts)} hexagons.")
        print(f"Point count range per hexagon: {self.min_count}-{self.max_count}.")
        return self.hex_counts

    def create_visualization(
        self,
        points: Optional[List[Tuple[float, float]]] = None,
        title: str = "H3 Grid Analysis",
        include_heatmap: bool = True,
        include_markers: bool = True
    ) -> folium.Map:
        """
        Generates a complete Folium map visualization with H3 hexagons,
        optional heatmap, and optional point clusters.

        Args:
            points: An optional list of (latitude, longitude) tuples to visualize.
                    If None, sample data will be generated.
            title: The title to display on the map legend.
            include_heatmap: If True, adds a heatmap layer.
            include_markers: If True, adds a FastMarkerCluster layer for individual points.

        Returns:
            A Folium map object.
        """
        # Use provided points or generate sample data
        data_points = points if points is not None else self.generate_sample_data()
        hex_data = self.process_data(data_points)

        # Create the base map
        m = self.create_base_map(data_points)

        # Add layers based on configuration
        if include_heatmap:
            self.add_heatmap_layer(m, data_points)

        self.add_hexagon_layer(m, hex_data)

        if include_markers:
            self.add_marker_cluster(m, data_points)

        # Add the legend
        self.add_legend(m, title)

        # Add layer control for toggling layers
        folium.LayerControl(position='topright', collapsed=False).add_to(m)

        return m

In [ ]:
# # 1. Default visualization
# viz = H3MapVisualizer()
# m = viz.create_visualization()
# file_path = './output/map/h3MapVisualization - Default.html'
# m.save(file_path)


# # 2. Percentage-based coloring
# viz = H3MapVisualizer(color_scheme='percentage')
# m = viz.create_visualization()
# file_path = './output/map/h3MapVisualization - Percentage-based coloring.html'
# m.save(file_path)

# # 3. Custom thresholds
# custom_scheme = {
#     'type': 'thresholds',
#     'thresholds': [3, 9, 15],
#     'colors': ['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c'],
#     'labels': ['Light', 'Medium', 'Dense', 'Very Dense']
# }
# viz = H3MapVisualizer(color_scheme=custom_scheme)
# m = viz.create_visualization(include_heatmap=False)

# file_path = './output/map/h3MapVisualization - Custom thresholds.html'
# m.save(file_path)

# # # 4. High-resolution map
# # viz = H3MapVisualizer(resolution=9, zoom_start=12)
# # m = viz.create_visualization()

In [182]:
df_clean = clean_df_coordinates(df = dict_causeway['sp_active_customers_dim'], lat_col='Latitude', lon_col='Longitude', return_point = False)
points = clean_df_coordinates(df = df_clean, lat_col='Latitude', lon_col='Longitude', return_point = True)
# df_clean.shape

visualizer = H3MapVisualizer(resolution=7, zoom_start=4, color_scheme='percentage')

# Process data
hex_counts = visualizer.process_data(points)
# Print summary statistics
print(f"\nSummary:")
print(f"Total hexagons: {len(visualizer.hex_counts)}")
print(f"Average points per hexagon: {visualizer.hex_counts['point_count'].mean():.2f}")
print(f"Most dense hexagon has {visualizer.hex_counts['point_count'].max()} points")

Dropping rows with null coordinates...
Dataframe now contains 1,946 valid coordinates
Successfully converted 1946 valid coordinates to points
Processed 1946 points into 110 hexagons.
Point count range per hexagon: 1-220.

Summary:
Total hexagons: 110
Average points per hexagon: 17.69
Most dense hexagon has 220 points


In [183]:
def create_h3_visualization(
        self,
        points: Optional[List[Tuple[float, float]]] = None,
        base_map = None,
        title: str = "H3 Grid Analysis",
        include_heatmap: bool = True,
        include_markers: bool = True
    ) -> folium.Map:
        """
        Generates a complete Folium map visualization with H3 hexagons,
        optional heatmap, and optional point clusters.

        Args:
            points: An optional list of (latitude, longitude) tuples to visualize.
                    If None, sample data will be generated.
            title: The title to display on the map legend.
            include_heatmap: If True, adds a heatmap layer.
            include_markers: If True, adds a FastMarkerCluster layer for individual points.

        Returns:
            A Folium map object.
        """
        # Use provided points or generate sample data
        data_points = points if points is not None else self.generate_sample_data()
        hex_data = self.process_data(data_points)

        # Create the base map
        if base_map is None:
            m = self.create_base_map(data_points)
        else:
            m = base_map
            
        # Add layers based on configuration
        if include_heatmap:
            self.add_heatmap_layer(m, data_points)

        self.add_hexagon_layer(m, hex_data)

        if include_markers:
            self.add_marker_cluster(m, data_points)

        # Add the legend
        self.add_legend(m, title)

        # Add layer control for toggling layers
        folium.LayerControl(position='topright', collapsed=False).add_to(m)

        return m

In [186]:
with open('./input/map_lagos_wards.pkl', 'rb') as f:
    m = pickle.load(f)
    

m = create_h3_visualization(
        self = visualizer,
        points  = points,
        base_map= None,
        title = "OmniHub Apapa Lagos - CAUSEWAY: Grid Analysis",
        include_heatmap = True,
        include_markers = False
    )

# Save the map
file_path = './output/map/causeway_grid_analysis.html'
m.save(file_path)
print(f"Map saved as {file_path}")

Processed 1946 points into 110 hexagons.
Point count range per hexagon: 1-220.
Map saved as ./output/map/causeway_grid_analysis.html


## H3 Investigation

#### Safe H3 Operation

In [ ]:
# Example with comprehensive error handling and validation
def safe_h3_operations(lat, lng, resolution):
    """Safely perform H3 operations with validation (NEW API)"""

    # Validate inputs
    if not (-90 <= lat <= 90 and -180 <= lng <= 180):
        raise ValueError(f"Invalid coordinates: {lat}, {lng}")

    if not (0 <= resolution <= 15):
        raise ValueError(f"Invalid resolution: {resolution}")

    try:
        # Convert to H3
        h3_index = h3.latlng_to_cell(lat, lng, resolution)

        # Validate the resulting cell
        if not h3.is_valid_cell(h3_index):
            raise ValueError(f"Invalid H3 cell generated: {h3_index}")

        # Perform operations
        result = {
            'h3_index': h3_index,
            'center': h3.cell_to_latlng(h3_index),
            'boundary': h3.cell_to_boundary(h3_index),
            'neighbors': list(h3.grid_disk(h3_index, 1)),
            'resolution': h3.get_resolution(h3_index),
            'is_pentagon': h3.is_pentagon(h3_index),
            'area_km2': h3.cell_area(h3_index, 'km^2'),
            'base_cell': h3.get_base_cell_number(h3_index)
        }
        
        return result
        
    except Exception as e:
        raise RuntimeError(f"Error in H3 operations: {str(e)}")


In [ ]:
# #  Test safe operations
# try:
#     result = safe_h3_operations(37.7749, -122.4194, 5)
#     print("Safe operations successful:")
#     print(f"  H3 index: {result['h3_index']}")
#     for key in result.keys():
#         print(key, "  ", result[key])
    
# except:
#     pass

#### Density based Clustering

In [ ]:
def h3_density_clustering(points_df, resolution, min_points=5, distance_threshold=1):
    """Perform density-based clustering using H3 (NEW API)"""
    
    # Convert points to H3
    points_df = points_df.copy()
    points_df['h3_index'] = points_df.apply(
        lambda row: h3.latlng_to_cell(row['lat'], row['lng'], resolution), 
        axis=1
    )
    
    # Count points per cell
    cell_counts = points_df.groupby('h3_index').size().reset_index(name='point_count')
    
    # Find dense cells
    dense_cells = cell_counts[cell_counts['point_count'] >= min_points]['h3_index'].tolist()
    
    print(f'Number of dense clusters: {len(dense_cells)} with above >= {min_points} points')
    # Cluster adjacent dense cells
    clusters = []
    processed = set()
    
    for cell in dense_cells:
        if cell in processed:
            continue
        
        # Find connected dense cells using BFS
        cluster = set()
        queue = [cell]
        
        while queue:
            current = queue.pop(0)
            if current in processed:
                continue
            
            processed.add(current)
            cluster.add(current)
            
            # Get neighbors
            neighbors = h3.grid_disk(current, distance_threshold)
            for neighbor in neighbors:
                if neighbor in dense_cells and neighbor not in processed:
                    queue.append(neighbor)
        
        if cluster:
            clusters.append(cluster)
    
    # Assign cluster IDs to points
    cluster_map = {}
    for cluster_id, cluster_cells in enumerate(clusters):
        for cell in cluster_cells:
            cluster_map[cell] = cluster_id
    
    points_df['cluster_id'] = points_df['h3_index'].map(cluster_map)
    points_df['cluster_id'] = points_df['cluster_id'].fillna(-1)  # -1 for noise
    
    return points_df, clusters


In [ ]:
import numpy as np


# Example usage
np.random.seed(42)

# Generate clustered point data
cluster_centers = [(37.7749, -122.4194), (37.7849, -122.4094), (37.7649, -122.4294)]
clustered_points = []

for i, (center_lat, center_lng) in enumerate(cluster_centers):
    # Generate points around each center
    for _ in range(50):
        lat = center_lat + np.random.normal(0, 0.002)
        lng = center_lng + np.random.normal(0, 0.002)
        clustered_points.append({
            'lat': lat,
            'lng': lng,
            'point_id': len(clustered_points),
            'true_cluster': i
        })

# Add some noise points
for _ in range(20):
    lat = 37.7749 + np.random.uniform(-0.02, 0.02)
    lng = -122.4194 + np.random.uniform(-0.02, 0.02)
    clustered_points.append({
        'lat': lat,
        'lng': lng,
        'point_id': len(clustered_points),
        'true_cluster': -1
    })

points_df = pd.DataFrame(clustered_points)

In [ ]:
len(points_df)
print(points_df.describe())
print(points_df.value_counts('true_cluster'))
# points_df.sample(10)


In [ ]:
# Perform clustering
clustered_df, clusters = h3_density_clustering(points_df, resolution=9, min_points=3)

# Analyze results
print(f"Found {len(clusters)} clusters")
print("\nCluster summary:")
cluster_summary = clustered_df.groupby('cluster_id').agg({
    'point_id': 'count',
    'lat': 'mean',
    'lng': 'mean'
}).rename(columns={'point_id': 'point_count'})

print(cluster_summary) 

In [ ]:
clustered_df.groupby(['true_cluster','cluster_id'])['point_id'].size()#.reset_index()

#### Polygons or Area

In [ ]:
h3.__version__

In [ ]:
# Work with polygons (NEW API)
from h3 import LatLngPoly

# Define a polygon (lat, lng coordinates)
polygon_coords = [
    (37.7749, -122.4194),
    (37.7849, -122.4194),
    (37.7849, -122.4094),
    (37.7749, -122.4094),
    (37.7749, -122.4194)  # Close the polygon
]

# Create LatLngPoly object
polygon = LatLngPoly(polygon_coords)

# Convert polygon to H3 cells (NEW API)
resolution = 9
cells = h3.polygon_to_cells(polygon, resolution)
print(f"Polygon contains {len(cells)} cells at resolution {resolution}")

# Convert cells back to polygon (NEW API)
reconstructed = h3.cells_to_geo(cells)
print(f"Reconstructed polygon has {len(reconstructed)} parts")

# Calculate polygon area using H3 cells
total_area = sum(h3.cell_area(cell, 'km^2') for cell in cells)
print(f"Total polygon area: {total_area:.6f} km²")


#### Clustering Problems: Using H3 for Spatial Clustering

In [ ]:
import numpy as np
from collections import defaultdict

# Sample point data for clustering (NEW API)
np.random.seed(42)
points = pd.DataFrame({
    'lat': np.random.normal(37.7749, 0.03, 1000),
    'lng': np.random.normal(-122.4194, 0.03, 1000),
    'id': range(1000)
})

# Convert to H3 for clustering (NEW API)
resolution = 7
points['h3_index'] = points.apply(lambda row: h3.latlng_to_cell(row['lat'], row['lng'], resolution), axis=1)


##### Basic clustering: group by H3 cell

In [ ]:

# Basic clustering: group by H3 cell
clusters = points.groupby('h3_index').agg({
    'id': 'count',
    'lat': 'mean',
    'lng': 'mean'
}).rename(columns={'id': 'point_count'}).reset_index()

print("Cluster summary:")
print(len(clusters))
print(clusters.head())

##### Advanced clustering: merge adjacent high-density cells

In [ ]:
# Advanced clustering: merge adjacent high-density cells (NEW API)
def merge_adjacent_clusters(clusters, min_density=5):
    """Merge adjacent H3 cells with high point density"""
    high_density = clusters[clusters['point_count'] >= min_density]
    merged_clusters = []
    processed = set()

    for _, cluster in high_density.iterrows():
        h3_index = cluster['h3_index']

        if h3_index in processed:
            continue

        # Get neighbors using NEW API
        neighbors = h3.grid_disk(h3_index, 1)

        # Find neighboring high-density cells
        adjacent_dense = high_density[high_density['h3_index'].isin(neighbors)]

        if len(adjacent_dense) > 1:
            # Mark all cells in this cluster as processed
            for _, adj_cluster in adjacent_dense.iterrows():
                processed.add(adj_cluster['h3_index'])

            # Merge cluster
            merged_cluster = {
                'center_h3': h3_index,
                'total_points': adjacent_dense['point_count'].sum(),
                'cells_in_cluster': len(adjacent_dense),
                'avg_lat': adjacent_dense['lat'].mean(),
                'avg_lng': adjacent_dense['lng'].mean(),
                'cells': list(adjacent_dense['h3_index'])
            }
            merged_clusters.append(merged_cluster)

    return pd.DataFrame(merged_clusters)

merged = merge_adjacent_clusters(clusters = clusters, min_density=5)
print("\nMerged clusters:")
print(len(merged))
print(merged.head(3))


#### Hierarchical clustering using parent-child relationships

In [ ]:
# Hierarchical clustering using parent-child relationships (NEW API)
def hierarchical_clustering(points_df, base_resolution=10, parent_resolution=8):
    """Perform hierarchical clustering using H3 parent-child relationships"""

    # Convert points to base resolution
    points_df = points_df.copy()
    points_df['h3_base'] = points_df.apply(
        lambda row: h3.latlng_to_cell(row['lat'], row['lng'], base_resolution),
        axis=1
    )

    # Get parent cells
    points_df['h3_parent'] = points_df['h3_base'].apply(
        lambda x: h3.cell_to_parent(x, parent_resolution)
    )

    # Aggregate by parent
    parent_clusters = points_df.groupby('h3_parent').agg({
        'id': 'count',
        'lat': 'mean',
        'lng': 'mean',
        'h3_base': lambda x: list(x.unique())
    }).rename(columns={'id': 'total_points'}).reset_index()

    # Add parent geometry
    parent_clusters['parent_center'] = parent_clusters['h3_parent'].apply(
        lambda x: h3.cell_to_latlng(x)
    )
    parent_clusters['parent_boundary'] = parent_clusters['h3_parent'].apply(
        lambda x: h3.cell_to_boundary(x)
    )

    return parent_clusters

hierarchical = hierarchical_clustering(points_df=points, base_resolution=resolution, parent_resolution=resolution-1)
print("\nHierarchical clusters:")
print(f'Total Cluster: {hierarchical.shape[0]}')
print(hierarchical[['h3_parent', 'total_points', 'parent_center']].head())


-----------------

## Vizualization

In [ ]:
# !pip install geopandas geodatasets contextily

In [ ]:
import h3

import geopandas
import geodatasets
import contextily as cx
import matplotlib.pyplot as plt

In [ ]:
def plot_df(df, column=None, ax=None):
    "Plot based on the `geometry` column of a GeoPandas dataframe"
    df = df.copy()
    df = df.to_crs(epsg=3857)  # web mercator

    if ax is None:
        _, ax = plt.subplots(figsize=(8,8))
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    df.plot(
        ax=ax,
        alpha=0.5, edgecolor='k',
        column=column, categorical=True,
        legend=True, legend_kwds={'loc': 'upper left'},
    )
    cx.add_basemap(ax, crs=df.crs, source=cx.providers.CartoDB.Positron)
    
    
def plot_shape(shape, ax=None):
    df = geopandas.GeoDataFrame({'geometry': [shape]}, crs='EPSG:4326')
    plot_df(df, ax=ax)
    
def plot_cells(cells, ax=None):
    shape = h3.cells_to_h3shape(cells)
    plot_shape(shape, ax=ax)

In [ ]:
import folium
import h3
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import json
from folium.plugins import HeatMap
from shapely.geometry import Point, Polygon
import warnings
warnings.filterwarnings('ignore')

class H3NigeriaMapper:
    def __init__(self):
        self.map = None
        self.h3_resolution = 7  # Adjust based on your needs (6-9 typical for city-level)
        
    def create_base_map(self, center_lat=9.0820, center_lon=8.6753, zoom=6):
        """Create base map centered on Nigeria"""
        self.map = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=zoom,
            tiles='OpenStreetMap'
        )
        return self.map
    
    def load_sample_data(self, n_points=1000):
        """Generate sample data points across Nigeria"""
        # Nigeria approximate bounds
        lat_min, lat_max = 4.0, 14.0
        lon_min, lon_max = 2.5, 15.0
        
        np.random.seed(42)
        data = {
            'latitude': np.random.uniform(lat_min, lat_max, n_points),
            'longitude': np.random.uniform(lon_min, lon_max, n_points),
            'value': np.random.randint(1, 100, n_points),  # Some metric to aggregate
            'category': np.random.choice(['A', 'B', 'C'], n_points)
        }
        return pd.DataFrame(data)
    
    def points_to_h3(self, df, lat_col='latitude', lon_col='longitude'):
        """Convert lat/lon points to H3 hexagons"""
        df['h3_index'] = df.apply(
            lambda row: h3.latlng_to_cell(
                row[lat_col], row[lon_col], self.h3_resolution
            ), axis=1
        )
        return df
    
    def aggregate_by_h3(self, df, value_col='value'):
        """Aggregate points by H3 hexagon"""
        h3_stats = df.groupby('h3_index').agg({
            value_col: ['count', 'sum', 'mean'],
            'category': lambda x: x.mode().iloc[0] if not x.empty else 'Unknown'
        }).round(2)
        
        # Flatten column names
        h3_stats.columns = ['point_count', 'total_value', 'avg_value', 'dominant_category']
        h3_stats = h3_stats.reset_index()
        
        # Get hexagon boundaries
        h3_stats['geometry'] = h3_stats['h3_index'].apply(
            lambda x: Polygon(h3.cell_to_boundary(x))#, geo_json=True
        )
        
        return gpd.GeoDataFrame(h3_stats, geometry='geometry')
    
    def add_h3_hexagons(self, h3_gdf, color_col='point_count'):
        """Add H3 hexagons to map with color coding"""
        # Normalize values for color mapping
        min_val = h3_gdf[color_col].min()
        max_val = h3_gdf[color_col].max()
        
        def get_color(value):
            # Color scale from light to dark blue
            normalized = (value - min_val) / (max_val - min_val) if max_val > min_val else 0
            opacity = 0.3 + (normalized * 0.7)
            return f'rgba(0, 100, 200, {opacity})'
        
        for idx, row in h3_gdf.iterrows():
            # Convert geometry to GeoJSON format
            coords = [[list(coord) for coord in row.geometry.exterior.coords]]
            
            popup_html = f"""
            <div style="font-family: Arial; font-size: 12px;">
                <b>H3 Index:</b> {row.h3_index}<br>
                <b>Points:</b> {row.point_count}<br>
                <b>Total Value:</b> {row.total_value}<br>
                <b>Avg Value:</b> {row.avg_value}<br>
                <b>Category:</b> {row.dominant_category}
            </div>
            """
            
            folium.Polygon(
                locations=coords,
                popup=folium.Popup(popup_html, max_width=300),
                color='blue',
                weight=1,
                fill=True,
                fillColor=get_color(row[color_col]),
                fillOpacity=0.7
            ).add_to(self.map)
    
    def load_nigeria_boundaries(self, admin_level='state'):
        """Load Nigerian administrative boundaries from GeoJSON files"""
        import os
        
        base_path = "../input/geojson"
        
        try:
            if admin_level == 'lga':
                file_path = os.path.join(base_path, "GRID3_NGA_-_Operational_LGA_Boundaries.geojson")
                with open(file_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            
            elif admin_level == 'ward':
                file_path = os.path.join(base_path, "Nigeria_-_Ward_Boundaries.geojson")
                with open(file_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            
            elif admin_level == 'state':
                # Extract state boundaries from LGA data by dissolving
                lga_gdf = gpd.read_file(os.path.join(base_path, "GRID3_NGA_-_Operational_LGA_Boundaries.geojson"))
                
                # Assuming there's a state column in the LGA data
                state_col = None
                for col in lga_gdf.columns:
                    if 'state' in col.lower() or 'admin1' in col.lower():
                        state_col = col
                        break
                
                if state_col:
                    # Dissolve LGA boundaries to create state boundaries
                    states_gdf = lga_gdf.dissolve(by=state_col).reset_index()
                    return json.loads(states_gdf.to_json())
                else:
                    print("Warning: Could not find state column in LGA data")
                    return self._create_nigeria_outline()
            
            else:
                return self._create_nigeria_outline()
                
        except FileNotFoundError as e:
            print(f"Error loading {admin_level} boundaries: {e}")
            return self._create_nigeria_outline()
        except Exception as e:
            print(f"Error processing {admin_level} boundaries: {e}")
            return self._create_nigeria_outline()
    
    def inspect_geojson_structure(self):
        """Inspect the structure of your GeoJSON files to understand the data"""
        import os
        
        base_path = "../input/geojson"
        files = [
            "GRID3_NGA_-_Operational_LGA_Boundaries.geojson",
            "Nigeria_-_Ward_Boundaries.geojson"
        ]
        
        for filename in files:
            file_path = os.path.join(base_path, filename)
            try:
                print(f"\n=== {filename} ===")
                gdf = gpd.read_file(file_path)
                print(f"Shape: {gdf.shape}")
                print(f"Columns: {list(gdf.columns)}")
                print(f"CRS: {gdf.crs}")
                print("\nFirst few rows of properties:")
                print(gdf.drop('geometry', axis=1).head(2))
                
                # Show unique values for potential grouping columns
                for col in gdf.columns:
                    if col != 'geometry' and gdf[col].dtype == 'object':
                        unique_count = gdf[col].nunique()
                        if unique_count < 50:  # Only show if not too many unique values
                            print(f"\nUnique values in '{col}' ({unique_count}): {sorted(gdf[col].unique())[:10]}")
                
            except Exception as e:
                print(f"Error reading {filename}: {e}")
        
        return None
    
    def add_admin_boundaries(self, admin_levels=['state', 'lga', 'ward']):
        """Add Nigerian administrative boundaries as overlays"""
        
        # Layer control
        boundary_layers = {}
        
        for level in admin_levels:
            # Create feature group for this admin level
            fg = folium.FeatureGroup(name=f'{level.upper()} Boundaries')
            
            # Load boundaries
            boundary_data = self.load_nigeria_boundaries(level)
            
            # Style configuration for different admin levels
            style_config = {
                'state': {'color': 'red', 'weight': 3, 'opacity': 0.8},
                'lga': {'color': 'orange', 'weight': 2, 'opacity': 0.7},
                'ward': {'color': 'yellow', 'weight': 1, 'opacity': 0.6}
            }
            
            # Function to create popup content
            def create_popup(feature):
                props = feature.get('properties', {})
                
                # Try to find name field (common variations)
                name_fields = ['name', 'Name', 'NAME', 'admin_name', 'lga_name', 'ward_name', 'ADM1_EN', 'ADM2_EN', 'ADM3_EN']
                name = 'Unknown'
                for field in name_fields:
                    if field in props:
                        name = props[field]
                        break
                
                # Create popup content with available properties
                popup_content = f"<b>{level.upper()}:</b> {name}<br>"
                
                # Add additional properties if available
                for key, value in props.items():
                    if key not in name_fields and len(str(value)) < 50:  # Avoid very long values
                        popup_content += f"<b>{key}:</b> {value}<br>"
                
                return popup_content
            
            # Add to feature group
            folium.GeoJson(
                boundary_data,
                style_function=lambda feature, style=style_config[level]: {
                    'fillColor': 'transparent',
                    'color': style['color'],
                    'weight': style['weight'],
                    'opacity': style['opacity'],
                    'fillOpacity': 0.1
                },
                popup=folium.Popup(
                    lambda feature: create_popup(feature),
                    max_width=300
                ),
                tooltip=folium.Tooltip(
                    lambda feature: f"{level.upper()}: {self._get_feature_name(feature)}"
                )
            ).add_to(fg)
            
            boundary_layers[level] = fg
        
        return boundary_layers
    
    def _get_feature_name(self, feature):
        """Extract name from feature properties"""
        props = feature.get('properties', {})
        name_fields = ['name', 'Name', 'NAME', 'admin_name', 'lga_name', 'ward_name', 'ADM1_EN', 'ADM2_EN', 'ADM3_EN']
        
        for field in name_fields:
            if field in props:
                return props[field]
        
        return 'Unknown'
    
    def create_full_map(self, df=None, save_path='nigeria_h3_map.html'):
        """Create complete map with all components"""
        
        # Create base map
        self.create_base_map()
        
        # Load or use provided data
        if df is None:
            df = self.load_sample_data(5000)  # 5K points for demo
        
        # Convert to H3 and aggregate
        df_h3 = self.points_to_h3(df)
        h3_aggregated = self.aggregate_by_h3(df_h3)
        
        # Add H3 hexagons
        self.add_h3_hexagons(h3_aggregated)
        
        # Add administrative boundaries
        boundary_layers = self.add_admin_boundaries()
        
        # Add boundary layers to map
        for layer_name, layer in boundary_layers.items():
            layer.add_to(self.map)
        
        # Add layer control
        folium.LayerControl().add_to(self.map)
        
        # Add legend
        legend_html = '''
        <div style="position: fixed; 
                    bottom: 50px; left: 50px; width: 200px; height: 120px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <h4>Legend</h4>
        <p><i class="fa fa-square" style="color:blue"></i> H3 Hexagons (Point Density)</p>
        <p><i class="fa fa-square" style="color:red"></i> State Boundaries</p>
        <p><i class="fa fa-square" style="color:orange"></i> LGA Boundaries</p>
        <p><i class="fa fa-square" style="color:yellow"></i> Ward Boundaries</p>
        </div>
        '''
        self.map.get_root().html.add_child(folium.Element(legend_html))
        
        # Save map
        self.map.save(save_path)
        print(f"Map saved to {save_path}")
        
        return self.map, h3_aggregated

# Example usage for clustering points WITHIN hexagons
def cluster_points_within_hexagons(df, h3_resolution=7):
    """
    Example of clustering points within H3 hexagons using DBSCAN
    This is more advanced and computationally intensive
    """
    from sklearn.cluster import DBSCAN
    
    # Convert to H3
    df['h3_index'] = df.apply(
        lambda row: h3.latlng_to_cell(row['latitude'], row['longitude'], h3_resolution), 
        axis=1
    )
    
    results = []
    
    # For each hexagon, cluster points within it
    for h3_idx, group in df.groupby('h3_index'):
        if len(group) < 2:  # Need at least 2 points to cluster
            continue
            
        # Prepare coordinates for clustering
        coords = group[['latitude', 'longitude']].values
        
        # Apply DBSCAN clustering
        clustering = DBSCAN(eps=0.01, min_samples=2).fit(coords)
        
        # Add cluster labels
        group = group.copy()
        group['cluster_id'] = clustering.labels_
        group['h3_cluster'] = group.apply(
            lambda row: f"{row['h3_index']}_{row['cluster_id']}", axis=1
        )
        
        results.append(group)
    
    return pd.concat(results, ignore_index=True)

    def _create_nigeria_outline(self):
        """Create a simple Nigeria country outline as fallback"""
        return {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[2.5, 4.0], [15.0, 4.0], [15.0, 14.0], [2.5, 14.0], [2.5, 4.0]]]
            },
            "properties": {"name": "Nigeria"}
        }



In [ ]:
mapper = H3NigeriaMapper()
# mapper.inspect_geojson_structure()

In [ ]:

# Usage example with inspection
if __name__ == "__main__":
    # Initialize mapper
    mapper = H3NigeriaMapper()
    
    # First, inspect your GeoJSON files to understand the structure
    print("Inspecting GeoJSON files...")
    # mapper.inspect_geojson_structure()
    
    print("\n" + "="*50)
    print("Creating map...")
    
    # Create the map
    nigeria_map, h3_data = mapper.create_full_map()
    
    # Print summary statistics
    print("H3 Aggregation Summary:")
    print(h3_data[['point_count', 'total_value', 'avg_value']].describe())
    
    